# Notebook 8: RAG Evaluation

## Project: Enterprise Document Intelligence Assistant using LLM and RAG

This is the eighth notebook of the project.

In the previous notebooks, we built the complete RAG pipeline:

User Question → Retrieval → Context → LLM → Answer

In this notebook, we will evaluate the system.

The goal of this notebook is to:

1. Load the FAISS index and chunk metadata.
2. Test retrieval using sample queries.
3. Measure retrieval latency.
4. Check whether retrieved chunks match the expected category.
5. Evaluate top-k retrieval quality.
6. Review generated RAG answers if available.
7. Save the evaluation results.

The output files from this notebook will be:

`retrieval_evaluation_results.csv`

`rag_answer_review.csv`

In [1]:
# Install required libraries
!pip install faiss-cpu sentence-transformers -q

# Import required libraries
import os
import time
import faiss
import pandas as pd
from sentence_transformers import SentenceTransformer


# Find input file automatically
def find_input_file(file_name, required=True):
    """
    Searches for a file in common Kaggle locations.

    Parameters:
        file_name (str): Name of the file to search
        required (bool): Whether to raise an error if file is missing

    Returns:
        str or None: File path if found, otherwise None
    """
    possible_paths = [
        file_name,
        f"/kaggle/working/{file_name}"
    ]

    for root, dirs, files in os.walk("/kaggle/input"):
        if file_name in files:
            possible_paths.append(os.path.join(root, file_name))

    for path in possible_paths:
        if os.path.exists(path):
            return path

    if required:
        raise FileNotFoundError(
            f"{file_name} not found. Please upload it as input to this notebook."
        )

    return None


# Load FAISS index and metadata
def load_retrieval_files(index_file, metadata_file):
    """
    Loads FAISS index and metadata.
    """
    index_path = find_input_file(index_file)
    metadata_path = find_input_file(metadata_file)

    index = faiss.read_index(index_path)
    metadata = pd.read_csv(metadata_path)

    print("Retrieval files loaded successfully.")
    print("FAISS index:", index_path)
    print("Metadata:", metadata_path)
    print("Number of vectors:", index.ntotal)
    print("Metadata shape:", metadata.shape)

    if index.ntotal != len(metadata):
        raise ValueError("FAISS index size does not match metadata rows.")

    return index, metadata


# Load embedding model
def load_embedding_model(model_name):
    """
    Loads the embedding model used for retrieval.
    """
    model = SentenceTransformer(model_name)

    print("Embedding model loaded successfully.")
    print("Model name:", model_name)

    return model


# Retrieve chunks
def retrieve_chunks(query, model, index, metadata, top_k=5):
    """
    Retrieves top-k chunks for a query and measures retrieval time.
    """
    start_time = time.time()

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    latency = time.time() - start_time

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        row = metadata.iloc[idx]

        results.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "chunk_id": row["chunk_id"],
            "doc_id": row["doc_id"],
            "title": row["title"],
            "category": row["category"],
            "chunk_text": row["chunk_text"],
            "latency_seconds": latency
        })

    return pd.DataFrame(results)


# Evaluate one query
def evaluate_query(query, expected_category, model, index, metadata, top_k=5):
    """
    Evaluates retrieval result for one query using category match.
    """
    results = retrieve_chunks(
        query=query,
        model=model,
        index=index,
        metadata=metadata,
        top_k=top_k
    )

    retrieved_categories = results["category"].astype(str).str.lower().tolist()
    expected_category = expected_category.lower()

    top_1_match = retrieved_categories[0] == expected_category
    top_3_match = expected_category in retrieved_categories[:3]
    top_5_match = expected_category in retrieved_categories[:5]

    return {
        "query": query,
        "expected_category": expected_category,
        "top_1_category": retrieved_categories[0],
        "top_1_match": top_1_match,
        "top_3_match": top_3_match,
        "top_5_match": top_5_match,
        "average_similarity_score": results["score"].mean(),
        "retrieval_latency_seconds": results["latency_seconds"].iloc[0]
    }


# Run full retrieval evaluation
def run_retrieval_evaluation(test_queries, model, index, metadata, top_k=5):
    """
    Runs evaluation for all test queries.
    """
    evaluation_rows = []

    for item in test_queries:
        result = evaluate_query(
            query=item["query"],
            expected_category=item["expected_category"],
            model=model,
            index=index,
            metadata=metadata,
            top_k=top_k
        )

        evaluation_rows.append(result)

    return pd.DataFrame(evaluation_rows)


# Review RAG answers if available
def review_rag_answers(file_name):
    """
    Loads sample RAG results from Notebook 7 if available.
    Adds simple manual-review columns.
    """
    rag_file_path = find_input_file(file_name, required=False)

    if rag_file_path is None:
        print("sample_rag_results.csv not found.")
        print("Skipping RAG answer review.")
        return None

    rag_df = pd.read_csv(rag_file_path)

    rag_df["answer_length_words"] = rag_df["answer"].astype(str).apply(
        lambda x: len(x.split())
    )

    rag_df["needs_manual_review"] = rag_df["answer_length_words"] < 5

    print("RAG results loaded successfully.")
    print("File:", rag_file_path)
    print("Shape:", rag_df.shape)

    return rag_df


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.8 MB/s eta 0:00:00:00:0100:01


In [2]:
# Load FAISS index and metadata
index_file = "bbc_faiss_index.index"
metadata_file = "bbc_chunk_metadata.csv"

faiss_index, metadata = load_retrieval_files(index_file, metadata_file)


# Preview metadata
print("\nMetadata Preview:")
display(metadata.head())

print("\nAvailable Categories:")
print(metadata["category"].value_counts())


# Load embedding model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = load_embedding_model(MODEL_NAME)


# Define test queries
test_queries = [
    {
        "query": "What is happening in football and sports events?",
        "expected_category": "sport"
    },
    {
        "query": "What are the latest updates in business and financial markets?",
        "expected_category": "business"
    },
    {
        "query": "What is happening in politics and government?",
        "expected_category": "politics"
    },
    {
        "query": "What are the latest technology and software updates?",
        "expected_category": "tech"
    },
    {
        "query": "What entertainment news is being discussed?",
        "expected_category": "entertainment"
    }
]

print("Number of test queries:", len(test_queries))


# Run retrieval evaluation
evaluation_df = run_retrieval_evaluation(
    test_queries=test_queries,
    model=embedding_model,
    index=faiss_index,
    metadata=metadata,
    top_k=5
)

print("\nRetrieval Evaluation Results:")
display(evaluation_df)


# Calculate overall evaluation metrics
top_1_accuracy = evaluation_df["top_1_match"].mean()
top_3_accuracy = evaluation_df["top_3_match"].mean()
top_5_accuracy = evaluation_df["top_5_match"].mean()
avg_latency = evaluation_df["retrieval_latency_seconds"].mean()
avg_score = evaluation_df["average_similarity_score"].mean()

print("\nOverall Retrieval Metrics")
print("Top-1 Category Accuracy:", round(top_1_accuracy, 3))
print("Top-3 Category Accuracy:", round(top_3_accuracy, 3))
print("Top-5 Category Accuracy:", round(top_5_accuracy, 3))
print("Average Similarity Score:", round(avg_score, 4))
print("Average Retrieval Latency:", round(avg_latency, 4), "seconds")


# Show detailed retrieval example
example_query = test_queries[0]["query"]

example_results = retrieve_chunks(
    query=example_query,
    model=embedding_model,
    index=faiss_index,
    metadata=metadata,
    top_k=5
)

print("\nDetailed Retrieval Example")
print("Query:", example_query)

display(example_results[[
    "rank",
    "score",
    "chunk_id",
    "doc_id",
    "title",
    "category"
]])

print("\nTop Retrieved Chunk Preview:")
print(example_results.iloc[0]["chunk_text"][:1000])


# Save retrieval evaluation results
retrieval_output_file = "retrieval_evaluation_results.csv"

evaluation_df.to_csv(retrieval_output_file, index=False)

print("\nRetrieval evaluation saved successfully.")
print("Output file:", retrieval_output_file)


# Review RAG answers from Notebook 7 if available
rag_review_df = review_rag_answers("sample_rag_results.csv")

if rag_review_df is not None:
    print("\nRAG Answer Review:")
    display(rag_review_df)

    rag_review_output_file = "rag_answer_review.csv"
    rag_review_df.to_csv(rag_review_output_file, index=False)

    print("\nRAG answer review saved successfully.")
    print("Output file:", rag_review_output_file)


# Final evaluation summary
summary = pd.DataFrame({
    "metric": [
        "Top-1 Category Accuracy",
        "Top-3 Category Accuracy",
        "Top-5 Category Accuracy",
        "Average Similarity Score",
        "Average Retrieval Latency Seconds"
    ],
    "value": [
        round(top_1_accuracy, 3),
        round(top_3_accuracy, 3),
        round(top_5_accuracy, 3),
        round(avg_score, 4),
        round(avg_latency, 4)
    ]
})

print("\nFinal Evaluation Summary:")
display(summary)

Retrieval files loaded successfully.
FAISS index: /kaggle/input/datasets/jahnavidulala/evaluation-input/bbc_faiss_index.index
Metadata: /kaggle/input/datasets/jahnavidulala/evaluation-input/bbc_chunk_metadata.csv
Number of vectors: 8622
Metadata shape: (8622, 7)

Metadata Preview:


,chunk_id,doc_id,chunk_index,title,category,chunk_text,chunk_word_count
0,1_1,1,1,Ukraine conflict: Your guide to understanding ...,unknown,More than 1.5 million Ukrainians have fled the...,21
1,2_1,2,1,Russian gymnast investigated for wearing pro-w...,unknown,Russian gymnast Ivan Kuliak is being investiga...,31
2,3_1,3,1,Ukraine crisis: The West fights back against P...,unknown,Several US presidents have failed to get the m...,21
3,4_1,4,1,Ukraine maps: New agreed ceasefire breaks down...,unknown,A ceasefire agreement in the southern city of ...,20
4,5_1,5,1,Man in dinghy in near miss with Southampton-bo...,unknown,The moment a man swims out of the path of a co...,20



Available Categories:
category
unknown    8622
Name: count, dtype: int64


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

,rank,score,chunk_id,doc_id,title,category
0,1,0.585764,7604_1,7604,The Hollywood Olympics: All you need to know a...,unknown
1,2,0.559270,7392_1,7392,Key rivalries to watch out for at Paris 2024,unknown
2,3,0.552960,2338_1,2338,World Cup 2022: What we learned from a group s...,unknown
3,4,0.540622,4334_1,4334,What we learned from the group stages,unknown
4,5,0.506012,5630_1,5630,PFA exploring legal action over increasing num...,unknown



Top Retrieved Chunk Preview:
The new sporting events, the venues, the stars, and the traffic - what to watch out for in four years.

Retrieval evaluation saved successfully.
Output file: retrieval_evaluation_results.csv
RAG results loaded successfully.
File: /kaggle/input/datasets/jahnavidulala/evaluation-input/sample_rag_results.csv
Shape: (5, 7)

RAG Answer Review:


,question,answer,top_source_title,top_source_category,top_similarity_score,answer_length_words,needs_manual_review
0,What are the main updates related to politics?,I do not have enough information in the provid...,Has frantic election campaign actually grapple...,unknown,0.483725,10,False
1,What is happening in sports news?,World Cup 2022: What we learned from a thrilli...,Key rivalries to watch out for at Paris 2024,unknown,0.517542,17,False
2,What are the key business and economy updates?,I do not have enough information in the provid...,BBC editors react to Sunak's 2022 Spring State...,unknown,0.472559,10,False
3,What technology-related news is discussed?,I do not have enough information in the provid...,Ros Atkins: My list of things I don't understa...,unknown,0.467551,10,False
4,What entertainment news is available?,I do not have enough information in the provid...,Ros Atkins: My list of things I don't understa...,unknown,0.466247,10,False



RAG answer review saved successfully.
Output file: rag_answer_review.csv

Final Evaluation Summary:


,metric,value
0,Top-1 Category Accuracy,0.0000
1,Top-3 Category Accuracy,0.0000
2,Top-5 Category Accuracy,0.0000
3,Average Similarity Score,0.4280
4,Average Retrieval Latency Seconds,0.0468
